# Homework — Insurance Charges vs BMI

Dataset: [Insurance](https://www.kaggle.com/datasets/mirichoi0218/insurance)

Goal: the dataset has known outliers, so per the assignment we focus on two columns — predict `charges`
from `bmi` with a simple linear regression, deal with the outliers, and report accuracy.

Task checklist: **A)** clean data · **B)** build model · **C)** measure accuracy · **D)** document (this notebook).

## A. Load & Clean the Data

In [1]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

pd.set_option("display.precision", 2)

In [2]:
raw = pd.read_csv("../data/insurance.csv")
raw.head()

,age,sex,bmi,children,smoker,region,charges
0,19,female,27.90,0,yes,southwest,16884.92
1,18,male,33.77,1,no,southeast,1725.55
2,28,male,33.00,3,no,southeast,4449.46
3,33,male,22.70,0,no,northwest,21984.47
4,32,male,28.88,0,no,northwest,3866.86


In [3]:
# Check for missing values / duplicates on the FULL row first -- checking only bmi + charges
# would risk flagging unrelated people who coincidentally share both numbers as "duplicates".
print("shape:", raw.shape)
print("missing values:\n", raw.isna().sum().sum(), "total")
print("duplicate rows (full record):", raw.duplicated().sum())

shape: (1338, 7)
missing values:
 0 total
duplicate rows (full record): 1


In [4]:
df = raw.drop_duplicates()[["bmi", "charges"]].copy()
df.describe()

,bmi,charges
count,1337.00,1337.00
mean,30.66,13279.12
std,6.10,12110.36
min,15.96,1121.87
25%,26.29,4746.34
50%,30.40,9386.16
75%,34.70,16657.72
max,53.13,63770.43


No missing values; one exact duplicate record removed. `describe()` already hints at outliers: `charges` has a max (~$63.8k) many times its median, and its std is larger than its mean — a classic long right tail.

## Spot the Outliers

In [5]:
box_figure = go.Figure()
box_figure.add_trace(go.Box(y=df["bmi"], name="bmi", marker_color="#2563eb"))
box_figure.update_layout(title="BMI — Boxplot", template="plotly_white", width=350, height=450)
box_figure.show()

box_figure2 = go.Figure()
box_figure2.add_trace(go.Box(y=df["charges"], name="charges", marker_color="#dc2626"))
box_figure2.update_layout(title="Charges — Boxplot", template="plotly_white", width=350, height=450)
box_figure2.show()

`charges` clearly has many points above the upper whisker (largely driven by smokers and other factors outside our two chosen columns). `bmi` has a milder set of high outliers. We'll remove rows that are outliers on *either* column using the standard IQR rule (1.5×IQR beyond Q1/Q3).

In [6]:
def drop_iqr_outliers(frame, columns, k=1.5):
    mask = pd.Series(True, index=frame.index)
    for col in columns:
        q1, q3 = frame[col].quantile([0.25, 0.75])
        iqr = q3 - q1
        lower, upper = q1 - k * iqr, q3 + k * iqr
        mask &= frame[col].between(lower, upper)
    return frame[mask]

clean_df = drop_iqr_outliers(df, ["bmi", "charges"])

print(f"rows before: {len(df)}")
print(f"rows after:  {len(clean_df)}")
print(f"rows removed as outliers: {len(df) - len(clean_df)}")

rows before: 1337
rows after:  1192
rows removed as outliers: 145


## Quick Look at the Cleaned Data

In [7]:
scatter = go.Figure()
scatter.add_trace(go.Scatter(x=df["bmi"], y=df["charges"], mode="markers",
                              marker=dict(size=6, color="#cbd5e1"), name="removed outliers"))
scatter.add_trace(go.Scatter(x=clean_df["bmi"], y=clean_df["charges"], mode="markers",
                              marker=dict(size=7, color="#2563eb"), name="kept"))
scatter.update_layout(title="Charges vs BMI — Outliers Highlighted", xaxis_title="BMI",
                       yaxis_title="Charges ($)", template="plotly_white", width=750, height=480)
scatter.show()

Even after trimming outliers the cloud stays wide — `bmi` alone barely relates to `charges` visually, since real-world charges are driven mostly by other factors (smoking status especially) that we intentionally left out of scope for this exercise.

## B. Build the Linear Regression Model

In [8]:
X = clean_df[["bmi"]]
y = clean_df["charges"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = LinearRegression()
model.fit(X_train, y_train)

print(f"slope (coef_):    {model.coef_[0]:.2f}")
print(f"intercept:        {model.intercept_:.2f}")

slope (coef_):    -54.69
intercept:        11429.48


In [9]:
line_x = np.linspace(X["bmi"].min() - 1, X["bmi"].max() + 1, 50)
line_y = model.predict(line_x.reshape(-1, 1))

fit_figure = go.Figure()
fit_figure.add_trace(go.Scatter(x=X_train["bmi"], y=y_train, mode="markers",
                                 marker=dict(size=7, color="#2563eb"), name="train"))
fit_figure.add_trace(go.Scatter(x=X_test["bmi"], y=y_test, mode="markers",
                                 marker=dict(size=7, color="#16a34a", symbol="diamond"), name="test"))
fit_figure.add_trace(go.Scatter(x=line_x, y=line_y, mode="lines",
                                 line=dict(color="#dc2626", width=3), name="fitted line"))
fit_figure.update_layout(title="Fitted Regression Line (outliers removed)", xaxis_title="BMI",
                          yaxis_title="Charges ($)", template="plotly_white", width=750, height=480)
fit_figure.show()

/home/akbar/akbarDev/hbai/academy-tutorials/linear-regression/.venv/lib/python3.13/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(


## C. Measure the Model's Accuracy

In [10]:
y_pred = model.predict(X_test)

r2 = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
rmse = mean_squared_error(y_test, y_pred) ** 0.5

print(f"R^2:  {r2:.3f}   (fraction of variance in charges explained by bmi)")
print(f"MAE:  ${mae:,.0f}   (average absolute prediction error)")
print(f"RMSE: ${rmse:,.0f}   (typical prediction error, penalizes big misses more)")

R^2:  -0.003   (fraction of variance in charges explained by bmi)
MAE:  $6,032   (average absolute prediction error)
RMSE: $7,644   (typical prediction error, penalizes big misses more)


In [11]:
# For comparison: same model/split logic, but on the *uncleaned* data (outliers left in).
Xo = df[["bmi"]]
yo = df["charges"]
Xo_train, Xo_test, yo_train, yo_test = train_test_split(Xo, yo, test_size=0.2, random_state=42)
outlier_model = LinearRegression().fit(Xo_train, yo_train)
yo_pred = outlier_model.predict(Xo_test)

comparison = pd.DataFrame({
    "Data": ["With outliers", "Outliers removed"],
    "R2": [r2_score(yo_test, yo_pred), r2],
    "MAE": [mean_absolute_error(yo_test, yo_pred), mae],
    "RMSE": [mean_squared_error(yo_test, yo_pred) ** 0.5, rmse],
})
comparison.round(3)

,Data,R2,MAE,RMSE
0,With outliers,5.20e-02,9891.12,13200.44
1,Outliers removed,-3.00e-03,6031.88,7644.15


In [12]:
print("correlation(bmi, charges)  -- with outliers:   ", df["bmi"].corr(df["charges"]).round(3))
print("correlation(bmi, charges)  -- outliers removed:", clean_df["bmi"].corr(clean_df["charges"]).round(3))

correlation(bmi, charges)  -- with outliers:    0.198
correlation(bmi, charges)  -- outliers removed: -0.061


The correlation flips from **weakly positive to weakly negative** once outliers are removed. That's not a bug in the cleaning step — it's because the biggest `charges` outliers in this dataset tend to be high-`bmi` people (real-world: obese smokers with very high claims). Those points carry most of the genuine positive bmi-charges signal. Removing them as "outliers" doesn't just remove noise here — it removes the story along with it.

## D. Summary

- **Data**: `bmi` and `charges` only, as scoped by the assignment. No missing values; one duplicate row
  dropped. Outliers on either column were flagged and removed with the IQR rule (1.5×IQR beyond Q1/Q3).
- **Model**: linear regression fit on the cleaned data, 80/20 train/test split.
- **Accuracy**: R² is essentially 0 after cleaning (see metrics above) — the model has no real linear
  predictive power once outliers are gone. MAE/RMSE stay in the thousands of dollars, similar to just guessing
  the mean.
- **Takeaway**: `bmi` alone was already a weak linear predictor of `charges` (raw correlation ≈ 0.2, R² ≈ 0.04).
  Removing outliers by the book made the fit *worse*, not better, because those "outliers" were largely
  high-bmi, high-charge people — the real relationship, not noise. The lesson: outlier removal isn't
  automatically an improvement; always check what you're throwing away before trusting the cleaned fit.